# Chapitre 11 · Bien s'entraîner

Notebook du chapitre 11 de *Construire un LLM de zéro*.

**Comment travailler.** Tout le code du chapitre est là, complet et prêt à exécuter :
on reprend le GPT du chapitre 10 (entraîné naïvement) et on l'améliore réglage par
réglage : AdamW, warmup + cosine, gradient clipping, précision mixte, accumulation
de gradient. Lis, exécute, triture. À la fin, la section **Exercices** : quatre
défis à trous, du plus simple au plus costaud, validés par des `assert`.

Tout tourne **sur CPU, hors ligne**. Aucun GPU requis.

In [ ]:
import math
import time
import json
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
print("PyTorch", torch.__version__, "| device : cpu (aucun GPU requis)")

## 0. On recharge le GPT du chapitre 10

Le corpus (les trente fables de La Fontaine), le tokeniseur caractère et la classe `GPT` viennent
tout droit du chapitre 10. On les recharge tels quels pour repartir exactement d'où on s'était arrêté.
Le corpus est lu depuis le notebook du chapitre 10 ; si le fichier est introuvable, un petit repli
inline prend le relais (le carnet ne dépend d'aucun réseau).

In [ ]:
def charger_corpus():
    """Lit le corpus des fables depuis le notebook du chapitre 10 ; repli inline sinon."""
    candidats = [
        Path("../partie_2_construire_le_cerveau/chapitre_10_le_transformer.ipynb"),
        Path("../../partie_2_construire_le_cerveau/chapitre_10_le_transformer.ipynb"),
    ]
    for chemin in candidats:
        if chemin.exists():
            nb = json.loads(chemin.read_text(encoding="utf-8"))
            for cellule in nb["cells"]:
                if cellule["cell_type"] == "code":
                    src = "".join(cellule["source"])
                    if src.strip().startswith("corpus"):
                        espace = {}
                        exec(src.split("chars =")[0], espace)
                        return espace["corpus"]
    # repli minimal (une fable) pour que le carnet tourne partout
    return ("LE CORBEAU ET LE RENARD\n"
            "Maitre corbeau, sur un arbre perche,\n"
            "Tenait en son bec un fromage.\n") * 200


corpus = charger_corpus()
chars = sorted(set(corpus))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in corpus])
print(f"corpus : {len(corpus)} caracteres | vocabulaire : {vocab_size} | tokens : {data.shape[0]}")

In [ ]:
block_size = 64


def fabriquer_batch(taille=32):
    ix = torch.randint(0, len(data) - block_size - 1, (taille,))
    x = torch.stack([data[i : i + block_size] for i in ix])          # (B, T)
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])  # (B, T), decale d'un cran
    return x, y


def decouper_en_tetes(X, n_heads):
    B, T, d_model = X.shape
    d_k = d_model // n_heads
    return X.view(B, T, n_heads, d_k).transpose(1, 2)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, d_model = x.shape
        Q = decouper_en_tetes(self.W_Q(x), self.n_heads)
        K = decouper_en_tetes(self.W_K(x), self.n_heads)
        V = decouper_en_tetes(self.W_V(x), self.n_heads)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask[:T, :T] == 0, float("-inf"))
        poids = torch.softmax(scores, dim=-1)
        out = (poids @ V).transpose(1, 2).contiguous().view(B, T, d_model)
        return self.W_O(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ffn(self.ln2(x))
        return x


d_model, n_heads, n_layers, d_ff = 96, 4, 2, 384


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.table_tokens = nn.Embedding(vocab_size, d_model)
        self.table_positions = nn.Embedding(block_size, d_model)
        self.blocs = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.ln_final = nn.LayerNorm(d_model)
        self.tete = nn.Linear(d_model, vocab_size, bias=False)
        self.register_buffer("masque", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T = x.shape
        h = self.table_tokens(x) + self.table_positions(torch.arange(T))
        for bloc in self.blocs:
            h = bloc(h, self.masque)
        return self.tete(self.ln_final(h))


n_params = sum(p.numel() for p in GPT().parameters())
print(f"GPT rechargé : {n_params} parametres (identique au chapitre 10)")

In [ ]:
def held_out_loss(modele, n=20, graine=999):
    """Loss moyenne sur n batchs frais (protocole de comparaison du chapitre 10)."""
    g = torch.Generator().manual_seed(graine)
    modele.eval()
    with torch.no_grad():
        pertes = []
        for _ in range(n):
            ix = torch.randint(0, len(data) - block_size - 1, (32,), generator=g)
            x = torch.stack([data[i : i + block_size] for i in ix])
            y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
            pertes.append(F.cross_entropy(modele(x).view(-1, vocab_size), y.view(-1)).item())
    modele.train()
    return sum(pertes) / len(pertes)

## 1. Le point de départ : l'entraînement naïf, poussé un cran trop loin

Au chapitre 10, on entraînait à `lr = 3e-3`, un réglage prudent. Un labo vise plus haut pour aller plus
vite. Montons le learning rate à `3e-2` (dix fois plus) et gardons la boucle naïve du chapitre 10 :
learning rate **constant**, `AdamW` par défaut, ni warmup ni clipping. On regarde ce qui se passe.

In [ ]:
STEPS = 800


def entrainer_naif(lr):
    torch.manual_seed(42)
    modele = GPT()
    opt = torch.optim.AdamW(modele.parameters(), lr=lr)     # constant, betas/wd par defaut
    t0 = time.time()
    for step in range(STEPS):
        x, y = fabriquer_batch()
        loss = F.cross_entropy(modele(x).view(-1, vocab_size), y.view(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % 100 == 0 or step == STEPS - 1:
            print(f"naif   | step {step:3d} | loss {loss.item():.2f}")
    print(f"-> {time.time() - t0:.0f} s | loss held-out {held_out_loss(modele):.2f}")
    return modele


gpt_naif = entrainer_naif(lr=3e-2)

La loss descend d'abord, puis se **scotche autour de 2.4** et n'en bouge plus. Le learning rate est trop
grand pour une boucle sans garde-fous : à chaque pas, le modèle sur-corrige et défait ce qu'il vient
d'apprendre. Rien ne plante, mais l'entraînement est bloqué. C'est exactement le genre de run qu'un
labo ne peut pas se permettre. Réparons-le, réglage par réglage.

## 2. AdamW : le bon optimiseur, bien réglé

On garde AdamW, mais on le règle comme les labos : `betas=(0.9, 0.95)` (au lieu de `0.999` par défaut,
pour réagir plus vite aux changements de gradient) et `weight_decay=0.1` (une régularisation forte,
découplée du pas adaptatif : c'est le W d'AdamW). Seul, ce réglage ne suffit pas encore ; c'est la
première brique de la recette.

In [ ]:
# On instancie juste pour montrer la signature ; l'entraînement complet arrive en section 7.
gpt_demo = GPT()
optimizer = torch.optim.AdamW(
    gpt_demo.parameters(),
    lr=3e-2,               # peak LR : le sommet du schedule (section 3)
    betas=(0.9, 0.95),     # beta2 = 0.95 (et non 0.999) : adaptation plus rapide
    weight_decay=0.1,      # weight decay fort, découplé (le W d'AdamW)
)
print("AdamW configuré :", optimizer.defaults)

Le weight decay n'est pas la seule arme contre l'overfitting. Le **dropout** éteint au hasard une
fraction des activations, à l'entraînement seulement : c'est `model.train()` qui l'active et
`model.eval()` qui le coupe. Notre petit GPT n'en a pas besoin (le weight decay suffit à le tenir),
mais regarde-le agir sur un tenseur de uns.

In [ ]:
drop = nn.Dropout(0.1)     # éteint 10 % des activations, à l'entraînement seulement

t = torch.ones(2, 8)
drop.train()               # mode entraînement : le dropout est actif
print("train :", drop(t)[0])   # des zéros au hasard, le reste rescalé en 1/0.9
drop.eval()                # mode lecture : le dropout est coupé
print("eval  :", drop(t)[0])   # tout passe, intact

## 3. Le learning rate qui s'adapte : warmup + cosine

Le learning rate ne devrait pas rester constant. On veut de petits pas au tout début (les poids sont
aléatoires, les gradients bruités), un grand pas au milieu, puis de tout petits pas à la fin pour
affiner. C'est le schedule **warmup + cosine**, le standard de GPT-3 et LLaMA. On le code à la main.

In [ ]:
def warmup_cosine(step, warmup_steps, total_steps, base_lr, min_lr=0.0):
    if step < warmup_steps:
        # Phase 1 : montee lineaire de 0 a base_lr
        return base_lr * step / warmup_steps
    # Phase 2 : descente en cosinus de base_lr vers min_lr
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress))


# Verifions les points-cles du schedule qu'on utilisera (peak 3e-2, min 3e-3, warmup 100 / 800 steps)
for step in [0, 50, 100, 450, 799]:
    lr = warmup_cosine(step, warmup_steps=100, total_steps=800, base_lr=3e-2, min_lr=3e-3)
    print(f"step {step:3d} | lr {lr:.5f}")

Le learning rate part de 0, grimpe linéairement jusqu'au sommet `3e-2` au step 100 (fin du warmup),
puis redescend en cosinus jusqu'à `3e-3` (10 % du sommet) à la fin. On branchera cette fonction sur
`LambdaLR`, le scheduler PyTorch qui accepte n'importe quelle fonction maison.

## 4. Gradient clipping : la ceinture de sécurité

AdamW est robuste : il borne tout seul la taille de ses pas, donc il diverge rarement jusqu'à `nan`.
Pour **voir** une vraie explosion, prenons l'optimiseur nu, `SGD`, avec un learning rate agressif
(`lr = 2.0`). Sans filet, un gradient énorme suffit à tout détruire. Le calcul est enveloppé dans un
`try/except` pour ne jamais casser l'exécution du carnet.

In [ ]:
def entrainer_sgd(lr, clip=None, steps=120):
    torch.manual_seed(42)
    modele = GPT()
    opt = torch.optim.SGD(modele.parameters(), lr=lr)
    journal = []
    for step in range(steps):
        x, y = fabriquer_batch()
        loss = F.cross_entropy(modele(x).view(-1, vocab_size), y.view(-1))
        opt.zero_grad()
        loss.backward()
        # clip_grad_norm_ renvoie la norme AVANT clipping : parfait pour l'observer
        norme = torch.nn.utils.clip_grad_norm_(modele.parameters(), clip if clip else 1e9)
        opt.step()
        if step in (0, 10, 20, 50, 119):
            journal.append((step, loss.item(), norme.item()))
    return journal


print("SANS clipping (SGD, lr=2.0) :")
try:
    for step, loss, norme in entrainer_sgd(lr=2.0, clip=None):
        etat = "nan" if loss != loss else f"{loss:.2f}"
        print(f"  step {step:3d} | loss {etat:>7} | norme du gradient {norme:.1f}")
except Exception as e:
    print("  exception attrapee :", type(e).__name__, e)

In [ ]:
print("AVEC clipping a 1.0 (meme SGD, meme lr=2.0) :")
for step, loss, norme in entrainer_sgd(lr=2.0, clip=1.0):
    print(f"  step {step:3d} | loss {loss:.2f} | norme AVANT clip {norme:.1f}")

Sans clipping, la norme du gradient bondit au-delà de 40 dès le step 10, la loss saute à plus de 30,
puis tout devient `nan` : l'entraînement est mort. Avec `clip_grad_norm_(..., 1.0)`, la même norme est
ramenée à 1.0 avant chaque pas ; la direction du gradient est conservée, seule son amplitude est bornée.
Le run survit et descend tranquillement. Une ligne, une ceinture de sécurité.

## 5. Précision mixte : entraîner en 16 bits

La précision mixte stocke et calcule certains nombres sur 16 bits au lieu de 32 : moitié moins de
mémoire, calcul plus rapide sur GPU. Le mot **mixte** est important : `autocast` ne bascule que les
opérations sûres (les grosses multiplications de matrices), et garde en fp32 celles qui sont sensibles
(les normalisations). On le vérifie de nos yeux, sur CPU, sans aucun GPU.

In [ ]:
x = torch.randn(4, 96)
lin = nn.Linear(96, 96)
ln = nn.LayerNorm(96)

with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
    y_matmul = lin(x)      # grosse multiplication de matrices
    y_norm = ln(x)         # normalisation

print("entree x           :", x.dtype)
print("apres matmul       :", y_matmul.dtype, "  <- bascule en bf16 (autocast)")
print("apres LayerNorm    :", y_norm.dtype, "   <- reste en fp32 (op sensible)")
print("hors autocast      :", lin(x).dtype, "  <- tout reste en fp32")

In [ ]:
# Empreinte memoire : bf16 divise par deux le poids d'un modele
for nom, n in [("notre GPT", n_params), ("GPT-2 small", 124_000_000)]:
    print(f"{nom:14s} : fp32 {n * 4 / 1e9:.3f} Go | bf16 {n * 2 / 1e9:.3f} Go (divise par 2)")

### 5.4 fp16 + GradScaler : la béquille du T4 de Colab

Sur le T4 gratuit de Colab, bf16 n'est pas disponible : on entraîne en fp16, dont la plage étroite
écrase les petits gradients à zéro. La parade est le `GradScaler` : il multiplie la loss avant le
`backward` pour gonfler les gradients dans la plage utile, puis les redivise avant le pas. La cellule
suivante ne s'exécute **que sur GPU CUDA** ; sur CPU (ici), elle s'affiche sans rien lancer.

In [ ]:
if torch.cuda.is_available():
    from torch.amp import autocast, GradScaler

    modele = GPT().cuda()
    opt = torch.optim.AdamW(modele.parameters(), lr=3e-3)
    scaler = GradScaler()  # utile uniquement en fp16
    x, y = fabriquer_batch()
    x, y = x.cuda(), y.cuda()

    with autocast(device_type="cuda", dtype=torch.float16):
        loss = F.cross_entropy(modele(x).view(-1, vocab_size), y.view(-1))

    scaler.scale(loss).backward()      # gonfle les gradients
    scaler.unscale_(opt)               # les redivise AVANT de clipper
    torch.nn.utils.clip_grad_norm_(modele.parameters(), 1.0)
    scaler.step(opt)                   # applique le pas si aucun inf/nan
    scaler.update()                    # ajuste le facteur d'echelle
    print("fp16 + GradScaler : un pas effectue sur GPU CUDA")
else:
    print("Pas de GPU CUDA ici : cellule fp16+GradScaler non executee (elle est faite pour le T4 de Colab).")
    print("En bf16, le GradScaler est INUTILE : la plage de bf16 est deja assez large.")

## 6. Accumulation de gradient : simuler un gros batch

Les gros modèles veulent de gros batchs (gradients moins bruités). Mais un petit GPU plafonne la taille
de batch. L'astuce : faire plusieurs forward + backward **sans** remettre les gradients à zéro (PyTorch
les additionne tout seul), puis un seul `optimizer.step()`. On vérifie que c'est **exactement** équivalent
à un gros batch.

In [ ]:
torch.manual_seed(0)
petit = nn.Linear(10, 3)
X = torch.randn(8, 10)
Y = torch.randint(0, 3, (8,))

# A) un seul batch de 8
petit.zero_grad()
F.cross_entropy(petit(X), Y).backward()
grad_gros_batch = petit.weight.grad.clone()

# B) 4 micro-batchs de 2, loss divisee par 4, gradients accumules
grand = nn.Linear(10, 3)
grand.load_state_dict(petit.state_dict())
grand.zero_grad()
accumulation_steps = 4
for i in range(accumulation_steps):
    xb, yb = X[i * 2:(i + 1) * 2], Y[i * 2:(i + 1) * 2]
    (F.cross_entropy(grand(xb), yb) / accumulation_steps).backward()  # on MOYENNE
grad_accumule = grand.weight.grad.clone()

print("gradients identiques :", torch.allclose(grad_gros_batch, grad_accumule, atol=1e-6))
print("ecart maximal        :", (grad_gros_batch - grad_accumule).abs().max().item())

`allclose == True` : accumuler 4 micro-batchs de 2 (en divisant la loss par 4) donne le **même** gradient
qu'un seul batch de 8. La mémoire ne voit jamais plus de 2 exemples à la fois. Sans la division par
`accumulation_steps`, les gradients seraient 4 fois trop grands.

## 7. Tout réuni : la recette du labo contre l'entraînement naïf

On rassemble tout dans une seule boucle : AdamW bien réglé, warmup + cosine, gradient clipping.
(La précision mixte et l'accumulation n'apportent rien sur CPU à cette échelle, on les garde pour Colab.)
Même GPT, même corpus, **même learning rate `3e-2`** que le run naïf de la section 1. On compare.

In [ ]:
def entrainer_labo(lr, warmup_steps=100):
    torch.manual_seed(42)
    modele = GPT()
    opt = torch.optim.AdamW(modele.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lr_lambda=lambda s: warmup_cosine(s, warmup_steps, STEPS, 1.0, 0.1)  # relatif au peak
    )
    t0 = time.time()
    for step in range(STEPS):
        x, y = fabriquer_batch()
        loss = F.cross_entropy(modele(x).view(-1, vocab_size), y.view(-1))
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(modele.parameters(), 1.0)   # gradient clipping
        opt.step()
        sched.step()                                               # avance le schedule
        if step % 100 == 0 or step == STEPS - 1:
            print(f"labo   | step {step:3d} | loss {loss.item():.2f} | lr {sched.get_last_lr()[0]:.4f}")
    print(f"-> {time.time() - t0:.0f} s | loss held-out {held_out_loss(modele):.2f}")
    return modele


gpt_labo = entrainer_labo(lr=3e-2)

In [ ]:
print("Le match, au meme learning rate 3e-2 :")
print(f"  naif (aucun reglage) : loss held-out {held_out_loss(gpt_naif):.2f}")
print(f"  recette du labo      : loss held-out {held_out_loss(gpt_labo):.2f}")

La même architecture, le même corpus, le même learning rate. Le run naïf reste scotché ; la recette du
labo descend franchement. La différence, ce ne sont pas les briques du modèle : ce sont les réglages
d'entraînement. C'est tout le message du chapitre. Dernier juge de paix : on fait écrire les deux
modèles, avec le même prompt.

In [ ]:
@torch.no_grad()
def generer(modele, prompt="Le ", n=180, temperature=0.8):
    modele.eval()
    idx = torch.tensor([[stoi[c] for c in prompt]])
    for _ in range(n):
        logits = modele(idx[:, -block_size:])[:, -1, :] / temperature
        probs = torch.softmax(logits, dim=-1)
        idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
    modele.train()
    return "".join(itos[i] for i in idx[0].tolist())


torch.manual_seed(0)
print("=== GPT naif (scotche) ===")
print(generer(gpt_naif))
print("\n=== GPT recette du labo ===")
print(generer(gpt_labo))

### 7.2 La config complète, telle qu'un labo l'écrit

Voici la recette réunie, celle qu'on relira en Partie III et IV. Sur un vrai GPU, on ajouterait
`autocast(bfloat16)` autour du forward et l'accumulation autour du step ; sur CPU on les laisse de côté.
Chaque ligne a une raison d'être, et tu les connais maintenant toutes.

In [ ]:
# Recette de reference (pseudo-boucle commentee, non executee ici : c'est la carte du chapitre)
recette = """
optimizer = torch.optim.AdamW(model.parameters(),
                              lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)   # ch.11 §2
scheduler = LambdaLR(optimizer, lr_lambda=warmup_cosine)                     # ch.11 §3

for step, batch in enumerate(loader):
    with autocast(device_type='cuda', dtype=torch.bfloat16):                 # ch.11 §5
        loss = model(batch).loss
    (loss / accumulation_steps).backward()                                   # ch.11 §6
    if (step + 1) % accumulation_steps == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)              # ch.11 §4
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
"""
print(recette)

## Exercices

À toi de jouer : quatre exercices, du plus simple (●) au plus costaud (●●●), pour réécrire
toi-même les gestes-clés du chapitre. Chaque cellule marquée `# TODO(toi)` contient un trou ;
complète-le, puis exécute la cellule de validation (`assert`) qui suit : si elle passe sans
erreur, c'est gagné.

**Le pacte « IA débranchée » agit ici** : pas d'IA pour remplir les trous à ta place. Elle peut
t'expliquer une erreur ; les doigts sur le clavier, c'est toi. Les réponses sont dans le notebook
solution, à n'ouvrir qu'après avoir vraiment essayé.

### Exercice 1 · Gradient clipping — niveau ●

Ajoute le gradient clipping dans la boucle SGD ci-dessous. La ligne manquante,
`torch.nn.utils.clip_grad_norm_(modele.parameters(), 1.0)`, va **entre** `loss.backward()` et
`opt.step()`. Elle renvoie la norme du gradient *avant* écrêtage (pratique pour l'observer).

In [ ]:
def entrainer_sgd_exo(lr, avec_clipping, steps=120):
    torch.manual_seed(42)
    modele = GPT()
    opt = torch.optim.SGD(modele.parameters(), lr=lr)
    journal = []
    for step in range(steps):
        x, y = fabriquer_batch()
        loss = F.cross_entropy(modele(x).view(-1, vocab_size), y.view(-1))
        opt.zero_grad()
        loss.backward()
        if avec_clipping:
            norme = None  # TODO(toi) : clippe la norme des gradients à 1.0 et récupère la norme
        else:
            norme = torch.nn.utils.clip_grad_norm_(modele.parameters(), 1e9)  # ne clippe rien
        opt.step()
        if step in (0, 10, 119):
            journal.append((step, loss.item(), norme.item()))
    return journal

In [ ]:
# Validation : gradient clipping.
sans = entrainer_sgd_exo(lr=2.0, avec_clipping=False)
avec = entrainer_sgd_exo(lr=2.0, avec_clipping=True)
print("sans clipping :", [(s, round(l, 1) if l == l else "nan") for s, l, _ in sans])
print("avec clipping :", [(s, round(l, 2)) for s, l, _ in avec])

# sans clipping, la loss finit en nan ; avec, elle reste finie et basse
assert sans[-1][1] != sans[-1][1], "sans clipping (lr=2.0), la loss doit exploser en nan"
assert avec[-1][1] < 3.5, "avec clipping, le run doit rester stable et descendre"
print("Exercice 1 OK : le clipping sauve un run qui, sans lui, diverge")

### Exercice 2 · Le schedule warmup + cosine — niveau ●●

Complète `warmup_cosine_exo`. Deux phases :
- si `step < warmup_steps` : montée **linéaire** de 0 à `base_lr` (donc `base_lr * step / warmup_steps`) ;
- sinon : descente en **cosinus** de `base_lr` vers `min_lr`, avec
  `progress = (step - warmup_steps) / (total_steps - warmup_steps)` et
  `min_lr + 0.5 * (base_lr - min_lr) * (1 + cos(pi * progress))`.

In [ ]:
def warmup_cosine_exo(step, warmup_steps, total_steps, base_lr, min_lr=0.0):
    if step < warmup_steps:
        return None  # TODO(toi) : montée linéaire de 0 à base_lr
    progress = None  # TODO(toi) : fraction d'avancement de la phase cosine
    return None      # TODO(toi) : la formule du cosine decay

In [ ]:
# Validation : le schedule warmup + cosine.
assert abs(warmup_cosine_exo(0, 100, 800, 3e-2, 3e-3) - 0.0) < 1e-9, "au step 0, lr doit valoir 0"
assert abs(warmup_cosine_exo(50, 100, 800, 3e-2, 3e-3) - 1.5e-2) < 1e-9, "à mi-warmup, lr = base_lr/2"
assert abs(warmup_cosine_exo(100, 100, 800, 3e-2, 3e-3) - 3e-2) < 1e-9, "fin du warmup : lr = peak"
assert abs(warmup_cosine_exo(799, 100, 800, 3e-2, 3e-3) - 3e-3) < 1e-4, "fin : lr ~ min_lr"
print("Exercice 2 OK : schedule warmup + cosine correct")

### Exercice 3 · Accumulation de gradient — niveau ●●

Complète la boucle d'accumulation : pour chaque micro-batch, calcule la loss, **divise-la** par
`accumulation_steps` (pour moyenner, pas sommer), puis `backward()`. Ne remets **pas** les gradients à
zéro entre les micro-batchs : PyTorch les additionne, c'est le but.

In [ ]:
torch.manual_seed(0)
petit = nn.Linear(10, 3)
X = torch.randn(8, 10)
Y = torch.randint(0, 3, (8,))

# reference : un seul batch de 8
petit.zero_grad()
F.cross_entropy(petit(X), Y).backward()
grad_gros_batch = petit.weight.grad.clone()

# accumulation : 4 micro-batchs de 2
grand = nn.Linear(10, 3)
grand.load_state_dict(petit.state_dict())
grand.zero_grad()
accumulation_steps = 4
for i in range(accumulation_steps):
    xb, yb = X[i * 2:(i + 1) * 2], Y[i * 2:(i + 1) * 2]
    # TODO(toi) : loss = cross_entropy(...) / accumulation_steps, puis loss.backward()
    pass

In [ ]:
# Validation : accumulation de gradient.
grad_accumule = grand.weight.grad
assert grad_accumule is not None, "tu n'as pas encore fait de backward"
assert torch.allclose(grad_gros_batch, grad_accumule, atol=1e-6), \
    "l'accumulation doit donner le MÊME gradient qu'un gros batch (as-tu divisé par accumulation_steps ?)"
print("Exercice 3 OK : accumuler 4 micro-batchs == un batch 4x plus gros")

### Exercice 4 · La recette du labo, réunie — niveau ●●●

Complète la boucle d'entraînement : gradient clipping à 1.0 après le `backward`, puis `opt.step()`
et `sched.step()`. Ton run doit descendre nettement plus bas que le naïf (plateau ~2.4).

In [ ]:
def entrainer_labo_exo(lr=3e-2, warmup_steps=100):
    torch.manual_seed(42)
    modele = GPT()
    opt = torch.optim.AdamW(modele.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lr_lambda=lambda s: warmup_cosine(s, warmup_steps, STEPS, 1.0, 0.1)
    )
    for step in range(STEPS):
        x, y = fabriquer_batch()
        loss = F.cross_entropy(modele(x).view(-1, vocab_size), y.view(-1))
        opt.zero_grad()
        loss.backward()
        # TODO(toi) : clippe à 1.0, puis opt.step(), sched.step()
        pass
        if step % 200 == 0 or step == STEPS - 1:
            print(f"step {step:3d} | loss {loss.item():.2f} | lr {sched.get_last_lr()[0]:.4f}")
    return modele


gpt_labo_exo = entrainer_labo_exo()

In [ ]:
# Validation : la recette du labo doit finir bien en dessous du plateau naïf (~2.4).
g = torch.Generator().manual_seed(999)
gpt_labo_exo.eval()
with torch.no_grad():
    pertes = []
    for _ in range(20):
        ix = torch.randint(0, len(data) - block_size - 1, (32,), generator=g)
        x = torch.stack([data[i : i + block_size] for i in ix])
        y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
        pertes.append(F.cross_entropy(gpt_labo_exo(x).view(-1, vocab_size), y.view(-1)).item())
held = sum(pertes) / len(pertes)
print(f"loss held-out : {held:.2f}")
assert held < 1.8, "la recette du labo doit descendre bien sous le plateau naïf (~2.4)"
print("Exercice 4 OK : la recette du labo bat l'entraînement naïf, au même learning rate")

## Verdict

Quatre validations vertes : tu sais entraîner comme un labo. Même GPT, même corpus, même learning
rate, et la loss passe de 2.4 (naïf) à 1.2 (recette du labo) : la recette d'entraînement compte
autant que l'architecture.

Pour aller plus loin (facultatif) :

- Rejoue `entrainer_labo` en enlevant une brique à la fois (warmup, clipping, betas) et regarde la loss remonter.
- Sur un GPU (Colab), enveloppe le forward dans `torch.autocast(device_type="cuda", dtype=torch.bfloat16)`
  et mesure le gain de temps.

Prochain chapitre : **survivre à Colab**. Ta boucle marche ; il faut maintenant qu'elle survive à une
session qui coupe (checkpointing, reprise).